# Sentiment Analysis of Feedback
### Task 2 | Fine-Tuned DistilBERT + Logistic Regression
**Dataset:** Feedback — 4000 Unique Entries

---

## ⚠️ Step 1: Enable GPU (Colab Only)
1. `Runtime` → `Change runtime type`
2. Set Hardware accelerator → **T4 GPU**
3. Click **Save**

## 1. Install & Import Libraries

In [ ]:
!pip install -q transformers datasets accelerate tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import torch
import shutil
try:
    from google.colab import files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    DistilBertTokenizerFast, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset

print("✅ Libraries imported.")
device_status = "cuda" if torch.cuda.is_available() else "CPU (WARNING: No GPU detected)"
print(f"🚀 Device detected: {device_status}")

## 2. Upload Dataset
Please upload the feedback CSV file.

In [ ]:
if IS_COLAB:
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
else:
    # Local path fallback
    DATA_PATH = "../data/internship_feedback_4000_unique.csv"
print(f"✅ Loading data from: {DATA_PATH}")

## 3. Load & Preprocess Data

In [ ]:
df = pd.read_csv(DATA_PATH)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_feedback'] = df['feedback_text'].apply(clean_text)
label2id = {"negative": 0, "positive": 1}
id2label = {0: "negative", 1: "positive"}
df['label'] = df['sentiment_label'].map(label2id)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    df['clean_feedback'], df['label'], test_size=0.15, random_state=42, stratify=df['label']
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, random_state=42, stratify=y_train_full
)

print(f"✅ Split: {len(X_train)} Train | {len(X_val)} Val | {len(X_test)} Test")

## 4. Logistic Regression Baseline

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

print(f"✅ Logistic Regression Accuracy: {lr_acc:.4f}")

## 5. Fine-Tune DistilBERT

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_dict({'text': X_train.tolist(), 'label': y_train.tolist()})
val_ds   = Dataset.from_dict({'text': X_val.tolist(), 'label': y_val.tolist()})
test_ds  = Dataset.from_dict({'text': X_test.tolist(), 'label': y_test.tolist()})

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_val   = val_ds.map(tokenize_function, batched=True)
tokenized_test  = test_ds.map(tokenize_function, batched=True)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=2,
    id2label=id2label,
    label2id=label2id
).to(device_status if "cuda" in device_status else "cpu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

trainer.train()

## 6. Final Results & Insights

In [ ]:
eval_results = trainer.evaluate(tokenized_test)
print(f"✅ BERT Final Accuracy: {eval_results['eval_accuracy']:.4f}")

# Custom Predictions Example
custom_feedback = ["The mentorship was amazing.", "I hated the repetitive work."]
inputs = tokenizer(custom_feedback, return_tensors="pt", padding=True, truncation=True).to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=1)
    for text, p in zip(custom_feedback, preds):
        print(f"- {text}: {id2label[p.item()]}")